# 17. `Component State Change: Component <Q> is in the unavailable state (HWID=<N>)` (unix.hw, state_change.unavailable)

In [1]:
from common import *

shape, log = show(17)

log shape 2: unix.hw | state_change.unavailable | Component State Change: Component <Q> is in the unavailable state (HWID=<N>)
2,831 logs (0.7%) from 257 nodes, 2004-01-01 08:32:01 to 2006-04-21 20:53:15
log 738 at 2004-05-15 17:30:52: 350533 node-200 unix.hw state_change.unavailable 1084642252 1 Component State Change: Component \042alt0\042 is in the unavailable state (HWID=2538)


Could be not good. Let's see if this is recurring, or if it comes after something else.

In [7]:
timeline(log.node, log.ts - day, log.ts)

,first_ts,last_ts,count,component,state,template
0,2004-05-14 18:03:49,2004-05-14 18:03:49,1,unix.hw,state_change.unavailable,Component State Change: Component <Q> is in the unavailable state (HWID=<N>)
1,2004-05-14 18:03:49,2004-05-14 18:03:49,1,unix.hw,state_change.available,Component State Change: Component <Q> is in the available state (HWID=<N>)
2,2004-05-15 13:10:29,2004-05-15 13:10:29,1,unix.hw,state_change.unavailable,Component State Change: Component <Q> is in the unavailable state (HWID=<N>)
3,2004-05-15 13:10:31,2004-05-15 13:10:31,1,unix.hw,state_change.available,Component State Change: Component <Q> is in the available state (HWID=<N>)
4,2004-05-15 13:11:02,2004-05-15 13:11:02,1,unix.hw,state_change.unavailable,Component State Change: Component <Q> is in the unavailable state (HWID=<N>)
5,2004-05-15 13:11:03,2004-05-15 13:11:03,1,unix.hw,state_change.available,Component State Change: Component <Q> is in the available state (HWID=<N>)
6,2004-05-15 13:11:05,2004-05-15 13:11:05,1,unix.hw,state_change.unavailable,Component State Change: Component <Q> is in the unavailable state (HWID=<N>)
7,2004-05-15 13:11:10,2004-05-15 13:11:10,1,unix.hw,state_change.available,Component State Change: Component <Q> is in the available state (HWID=<N>)
8,2004-05-15 13:22:16,2004-05-15 13:22:16,1,unix.hw,state_change.unavailable,Component State Change: Component <Q> is in the unavailable state (HWID=<N>)
9,2004-05-15 13:22:19,2004-05-15 13:22:19,1,unix.hw,state_change.available,Component State Change: Component <Q> is in the available state (HWID=<N>)


This is recurring.
- The state changes on this node in the 2 hours before

In [8]:
flaps = as_frame(logs.search(log.ts - 2 * hour, log.ts, node=log.node, component="unix.hw", limit=None))
display(flaps[["ts", "state", "message"]])

,ts,state,message
0,2004-05-15 15:40:02,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=2538)
1,2004-05-15 15:40:07,state_change.available,Component State Change: Component \042alt0\042 is in the available state (HWID=2538)
2,2004-05-15 15:50:35,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=2538)
3,2004-05-15 15:50:36,state_change.available,Component State Change: Component \042alt0\042 is in the available state (HWID=2538)
4,2004-05-15 16:01:04,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=2538)
5,2004-05-15 16:01:04,state_change.available,Component State Change: Component \042alt0\042 is in the available state (HWID=2538)
6,2004-05-15 17:04:37,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=2538)
7,2004-05-15 17:04:37,state_change.available,Component State Change: Component \042alt0\042 is in the available state (HWID=2538)
8,2004-05-15 17:30:22,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=2538)
9,2004-05-15 17:30:22,state_change.available,Component State Change: Component \042alt0\042 is in the available state (HWID=2538)


Seconds from each `unavailable` to the line after it

In [3]:
went = flaps.state == "state_change.unavailable"
seconds_unavailable = (flaps.ts.shift(-1) - flaps.ts)[went].dt.total_seconds()
print(f"{went.sum()} times unavailable, for at most {seconds_unavailable.max():.0f} seconds")
assert (flaps.state.shift(-1)[went] == "state_change.available").all()

14 times unavailable, for at most 5 seconds


Looks like this alt0 component is under constant heavy load, just going down and back up again. It's not a complete long outage but it's problematic. This is a network interface. How many nodes are affected?
- Nodes with this shape in the last hour

In [16]:
past_day = as_frame(logs.search(log.ts - day, log.ts, log_shape_id=shape.log_shape_id))
display(past_day)
print(f"unique nodes: {past_day.node.nunique()}")

,id,ts,node,component,state,message,log_shape_id
0,886,2004-05-14 18:03:49,node-193,unix.hw,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=4432),2
1,887,2004-05-14 18:03:49,node-198,unix.hw,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=2748),2
2,888,2004-05-14 18:03:49,node-201,unix.hw,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=3098),2
3,889,2004-05-14 18:03:49,node-203,unix.hw,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=3238),2
4,890,2004-05-14 18:03:49,node-195,unix.hw,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=2958),2
...,...,...,...,...,...,...,...
163,737,2004-05-15 17:30:52,node-202,unix.hw,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=3168),2
164,738,2004-05-15 17:30:52,node-200,unix.hw,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=2538),2
165,739,2004-05-15 17:30:52,node-201,unix.hw,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=3098),2
166,740,2004-05-15 17:30:52,node-203,unix.hw,state_change.unavailable,Component State Change: Component \042alt0\042 is in the unavailable state (HWID=3238),2


unique nodes: 12


It's affecting 12 nodes. Do they become unreachable?

In [22]:
logs.search(log.ts - day, log.ts, log_shape_id=NOT_RESPONDING)

[]

Ok nothing going down, just busy network.
Is this a common pattern in this cluster?
- This shape per hour over the 30 days before

In [23]:
by_hour = per_hour(log.ts - 30 * day, log.ts, log_shape_id=shape.log_shape_id)
print(f"{by_hour.sum():.0f} lines, all inside {len(by_hour)} hours across the whole month")
by_hour.sort_index()

778 lines, all inside 22 hours across the whole month


2004-04-21 17      2.0
2004-04-24 08      5.0
2004-04-24 09      1.0
2004-04-24 10     12.0
2004-04-25 03     12.0
2004-04-28 23     33.0
2004-04-30 00     50.0
2004-04-30 01      8.0
2004-05-11 08    103.0
2004-05-11 09    128.0
2004-05-11 10    225.0
2004-05-11 17      3.0
2004-05-12 05      2.0
2004-05-12 13      1.0
2004-05-14 13     12.0
2004-05-14 17     13.0
2004-05-14 18     12.0
2004-05-15 13     48.0
2004-05-15 14     36.0
2004-05-15 15     24.0
2004-05-15 16     12.0
2004-05-15 17     36.0
dtype: float64

Do nodes crash after this type of pattern? I need to know if an outage might follow.

In [24]:
busiest_incident_start = datetime(2004, 5, 11, 8, 0, 0)
busiest_incident_end = datetime(2004, 5, 11, 11, 0, 0)
logs.search(busiest_incident_start, busiest_incident_end, log_shape_id=NOT_RESPONDING)

[]

# Verdict: Warning. 
#### The component is \"alt0\", a network interface, and each drop is back within seconds.
#### Looks like the network is busy but hasn't crashed, and when this happened in the past (the busiest time in the past month) it also didn't result in any crashes.
Crashes around would be critical, and so would a pattern of busy->crash in the past. But this is neither.